In [1]:
!pip install darts


     ---------------------------------------- 0.0/54.7 kB ? eta -:--:--
     ------- -------------------------------- 10.2/54.7 kB ? eta -:--:--
     ------- -------------------------------- 10.2/54.7 kB ? eta -:--:--
     -------------- ----------------------- 20.5/54.7 kB 131.3 kB/s eta 0:00:01
     --------------------- ---------------- 30.7/54.7 kB 146.3 kB/s eta 0:00:01
     ----------------------------------- -- 51.2/54.7 kB 201.8 kB/s eta 0:00:01
     -------------------------------------- 54.7/54.7 kB 202.9 kB/s eta 0:00:00
     ---------------------------------------- 0.0/169.6 kB ? eta -:--:--
     -------------------------------------- 169.6/169.6 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Using cached Cython-3.0.11-cp312-cp312-win_amd64.whl.metadata (3.2 kB)
   ---------------------------------------- 0.0/963.3 kB ? eta -:--:--
   ------------------------- ------------- 624.6/963.3 kB 13.1 


[notice] A new release of pip is available: 24.0 -> 25.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import datetime
import sklearn
from sklearn.metrics import mean_squared_error

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import KernelPCA
import numpy as np
import pandas as pd
import math
import keras
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler
from keras.models import Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, Flatten, Concatenate, TimeDistributed
from tensorflow.keras import Input

data=pd.read_csv("ML_training2.csv").iloc[:,1:]
unique_players=data["name"].unique()

def singleStepSampler(df, window,pred_val):
    xRes, yRes = [], []
    futureCovRes = []
    future_known_cols=[0,1]
    
    for i in range(0, len(df) - window):
        # Past data for features
        past_data = df.iloc[i:i + window, 2:].values  # Exclude the target column
        
        # Future known data for features
        future_data = df.iloc[i + window:i + window + 1, future_known_cols].values  # Future features for the next time step
        xRes.append(past_data)
        yRes.append(df.iloc[i + window][pred_val])  # Target value for the next time step
        futureCovRes.append(future_data)
        
    return np.array(xRes), np.array(yRes), np.array(futureCovRes)
    

All_X_train= []
All_y_train=[]
All_future_train=[]

All_X_test=[]
All_y_test=[]
All_future_test=[]
names=[]
scalers=[]
opp_xg = data.apply(lambda row: row[22] if row[18] else row[20], axis=1)
opp_xgc = data.apply(lambda row: row[23] if row[18] else row[21], axis=1)
data["opposition_xg"]=opp_xg
data["opposition_xgc"]=opp_xgc

for k in range(len(unique_players)):
    
    player=unique_players[k]
    df=data[data["name"]==player]
    df=df[df["season"]!=30]
    opp_xg = df.apply(lambda row: row[22] if row[18] else row[20], axis=1)
    opp_xgc = df.apply(lambda row: row[23] if row[18] else row[21], axis=1)
    df["opposition_xg"]=opp_xg
    df["opposition_xgc"]=opp_xgc
    df["opposition_xgc"]=df["opposition_xgc"].astype(float)
    df["Own_Attacking_form"]=df["Own_Attacking_form"].astype(float)
    df["minutes"]=df["minutes"].astype(float)

    columns=["opposition_xgc","Own_Attacking_form","minutes","rolling_XG_historic","Threat","expected_goals"]
    #columns=["opposition_xgc","Own_Attacking_form","minutes","XG_slope","Threat_slope","expected_goals"]
    pred_val="expected_goals"
    columns=["opposition_xgc","Own_Attacking_form","minutes","rolling_XA_historic","creativity","expected_assists"]
    pred_val="expected_assists"
    """
    columns=["opposition_xgc","Own_Attacking_form","minutes","ICT","rolling_bps_historic","bps"]
    columns=["opposition_xgc","Own_Attacking_form","minutes","ICT","total_points","bonus"]
    pred_val="bonus" """

    
    df=df[columns]

    columns_to_scale=columns
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaler.fit(data[columns_to_scale].to_numpy())
    df[columns_to_scale] = scaler.transform(df[columns_to_scale].to_numpy())

    max_time=len(df)
    
    window=8
    if(len(df)<window+9):
        continue
    names.append(player)
    scalers.append(scaler)
    (xVal, yVal,futureCovRes) = singleStepSampler(df, window,pred_val)


    X_train = xVal[:len(xVal)-8]
    y_train = yVal[:len(xVal)-8]
    future_train=futureCovRes[:len(xVal)-8]

    X_test = xVal[len(xVal)-8:]
    y_test = yVal[len(xVal)-8:]
    future_test=futureCovRes[len(xVal)-8:]
    
    All_X_train.extend(X_train)
    All_y_train.extend(y_train)
    All_future_train.extend(future_train)

    
    All_X_test.extend(X_test)
    All_y_test.extend(y_test)
    All_future_test.extend(future_test)

All_X_train = np.array(All_X_train)
All_y_train = np.array(All_y_train)
All_future_train = np.array(All_future_train)

All_X_test = np.array(All_X_test)
All_y_test = np.array(All_y_test)
All_future_test = np.array(All_future_test)


shuffle_indices = np.random.permutation(len(All_X_train))

# Apply the same permutation to all arrays
All_X_train = All_X_train[shuffle_indices]
All_y_train = All_y_train[shuffle_indices]
All_future_train = All_future_train[shuffle_indices]



input_past = Input(shape=(window, len(xVal[0][0])))  # Past data (window size, number of features)
input_future = Input(shape=(1, len(futureCovRes[0][0])))  # Future data (1 time step, number of future features)


#Model

lstm_out = LSTM(32, return_sequences=True)(input_past)
lstm_out = LSTM(32)(lstm_out)  # Another LSTM layer after the first
#lstm_out = LSTM(32)(input_past)
# Add Dropout for regularization
lstm_out = Dropout(0.1)(lstm_out)  # Dropout with 30% rate to prevent overfitting

# Process future data and flatten
future_flat = Flatten()(input_future)

# Merge the LSTM output with future covariates
merged = Concatenate()([lstm_out, future_flat])

# Additional Dense layers to learn complex relationships
dense_out = Dense(32, activation='relu')(merged)  # Add a Dense layer with ReLU activation
dense_out2 = Dense(16, activation='relu')(dense_out)
dense_out = Dropout(0.1)(dense_out2)  # Another Dropout layer to avoid overfitting

# Final output layer
output = Dense(1)(dense_out)  # Output layer (regression, so no activation) 




model = Model(inputs=[input_past, input_future], outputs=output)
model.compile(optimizer='adam', loss='mse')
model.fit([All_X_train, All_future_train], All_y_train, epochs=100, batch_size=32)

model.save("LSTM_Goals.h5")


MSE=[]

for y in range(len(All_X_test)//8):
    x_tests=All_X_test[y*8:y*8+8,:]
    futures=All_future_test[y*8:y*8+8,:]
    ys=All_y_test[y*8:y*8+8]
    predictions = model.predict([x_tests, futures])

    y_pred_rescaled = scalers[y].inverse_transform(
        np.concatenate((np.zeros((len(predictions), len(columns_to_scale) - 1)), predictions.reshape(-1, 1)), axis=1)
    )[:, -1]

    y_rescaled = scalers[y].inverse_transform(
        np.concatenate((np.zeros((len(ys), len(columns_to_scale) - 1)), ys.reshape(-1, 1)), axis=1)
    )[:, -1]

    preds=[]
    for i in range(len(y_pred_rescaled)):
        preds.append(y_pred_rescaled[i].item())
    print(names[y])
    y_vals=y_rescaled
    print(y_vals)
    MSE.append(mean_squared_error(y_vals, y_pred_rescaled))
    print(preds)
print(sum(MSE) / len(MSE))
#0.0297GOALS
#0.009655819138158217 ASSIST
#105.146602251051242-BPS
#0.35 _bonus

C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_22180\1507378935.py:51: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  opp_xg = data.apply(lambda row: row[22] if row[18] else row[20], axis=1)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_22180\1507378935.py:52: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  opp_xgc = data.apply(lambda row: row[23] if row[18] else row[21], axis=1)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_22180\1507378935.py:61: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (cons

Epoch 1/100
567/567 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - loss: 0.0076
Epoch 2/100
567/567 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 0.0065
Epoch 3/100
567/567 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0065
Epoch 4/100
567/567 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 0.0060
Epoch 5/100
567/567 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0068
Epoch 6/100
567/567 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.0064
Epoch 7/100
567/567 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.0063
Epoch 8/100
567/567 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.0065
Epoch 9/100
567/567 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.0062
Epoch 10/100
567/567 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 0.0064
Epoch 11/100
567/567 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 0.0064
Epoch 12/100
567/567 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 0.0063
Epoch 13/100
567/567 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.0062
Epoch 14/100
567/567 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 0.0064
Epoch 15/100
567/567 ━━━━━━━━━━━━━━━━━━━━ 3

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 341ms/step
Fábio_Ferreira Vieira
[0.   0.03 0.02 0.01 0.   0.23 0.01 0.05]
[0.09567410856485367, 0.07332125321030616, 0.07090319260954857, 0.0408517637103796, 0.05304537478834391, 0.04908645495772362, 0.05973359920084477, 0.05561829071491957]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
Gabriel_Fernando de Jesus
[0.05 0.   0.   0.02 0.01 0.1  0.14 0.13]
[0.03493773926049471, 0.06368847213685513, 0.04765141189098358, 0.05060925375670195, 0.037286630272865294, 0.06470853053033353, 0.06880873572081328, 0.08510105699300766]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
Gabriel_dos Santos Magalhães
[0.01 0.07 0.01 0.02 0.04 0.09 0.01 0.03]
[0.0631938036158681, 0.059703830704092985, 0.07316783264279365, 0.06667409464716911, 0.07005127474665641, 0.06467176347970963, 0.04893806152045727, 0.050993993505835535]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
Kai_Havertz0
[0.01 0.09 0.01 0.08 0.01 0.03 0.   0.25]
[0.08519393842667342, 0.08782738253474236, 0.07778138011693955, 0.100103875547647

In [133]:

from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import load_model
from tensorflow.keras.models import load_model
from tensorflow.keras.losses import MeanSquaredError


def Generate_LSTM_preds(pred):
    data=pd.read_csv("ML_training2.csv").iloc[:,1:]
    opp_xg = data.apply(lambda row: row[22] if row[18] else row[20], axis=1)
    opp_xgc = data.apply(lambda row: row[23] if row[18] else row[21], axis=1)
    data["opposition_xg"]=opp_xg
    data["opposition_xgc"]=opp_xgc
    unique_players=data["name"].unique()
    pred_all_players=pd.DataFrame(columns=["Name", "p1","p2","p3","p4","p5","p6","p7","p8","position"])
    window=8
    for k in range(len(unique_players)):
        pred_player_df=[]
        player_name=unique_players[k]
        
        pred_player_df.append(player_name)
        
        df=data[data["name"]==unique_players[k]]
        position=df["position"].values[-1]
        past_df=df[df["season"]!=30].sort_values(by="time", ascending=True)

        test_df=df[df["season"]==25]
        if(len(test_df)<1):
            continue
        
        if(pred=="GOALS"):
            past_columns=["minutes","shots","Threat","expected_goals"]
            past_columns=["minutes","rolling_XG_historic","Threat","expected_goals"]
            future_columns=["opposition_xgc","Own_Attacking_form"]
            pred_variable="expected_goals"
            model_path="LSTM_Goals.h5"
        elif(pred=="Assist"):
            past_columns=["minutes","rolling_XA_historic","creativity","expected_assists"]
            future_columns=["opposition_xgc","Own_Attacking_form"]
            pred_variable="expected_assists"
            model_path="LSTM_Assist.h5"
        elif(pred=="bps"):
            past_columns=["minutes","ICT","total_points","bonus"]
            future_columns=["opposition_xgc","Own_Attacking_form"]
            pred_variable="bonus"
            model_path="LSTM_Bonus.h5"



        past_df=past_df[past_columns]

        past_scaler = MinMaxScaler(feature_range=(0, 1))
        past_scaler.fit(data[past_columns].to_numpy())
        past_df[past_columns]=past_scaler.transform(past_df[past_columns].to_numpy())

        
        if len(past_df) < window:
            num_missing = window - len(past_df)
            zero_rows = pd.DataFrame(np.zeros((num_missing, past_df.shape[1])), columns=past_df.columns)
            past_df = pd.concat([zero_rows, past_df], ignore_index=True)

        opp_xg = df.apply(lambda row: row[22] if row[18] else row[20], axis=1)
        opp_xgc = df.apply(lambda row: row[23] if row[18] else row[21], axis=1)
        df["opposition_xg"]=opp_xg
        df["opposition_xgc"]=opp_xgc

        future_scaler = MinMaxScaler(feature_range=(0, 1))
        
        future_scaler.fit(data[future_columns].to_numpy())
        df[future_columns]=future_scaler.transform(df[future_columns].to_numpy())
    
        

        future_data=df[df["season"]==30].sort_values(by="time", ascending=True)
        future_data=future_data[future_columns]
        
        if len(future_data) < window:
            num_missing = window - len(future_data)
            zero_rows = pd.DataFrame(np.zeros((num_missing, future_data.shape[1])), columns=future_data.columns)
            future_data = pd.concat([zero_rows, future_data], ignore_index=True)

        
        future=len(future_data)

        past_df=past_df.iloc[-window:,:]

        X_test=[]
        X_future=[]
        for i in range(future): 
            X_test.append(past_df.values)
            X_future.append([future_data.values[i]])

        X_test = np.array(X_test)
        X_future = np.array(X_future)

        if(player_name=="Mohamed_Salah"):
            print(X_test)
            print(X_future)
        # Load the model
        model = load_model(model_path, compile=False)

        # Compile again with the correct loss function
        model.compile(optimizer='adam', loss=MeanSquaredError()) 


        predictions = model.predict([X_test, X_future])

        y_pred_rescaled = past_scaler.inverse_transform(
            np.concatenate((np.zeros((len(predictions), len(past_columns) - 1)), predictions.reshape(-1, 1)), axis=1)
        )[:, -1]

        for y in range(len(y_pred_rescaled)):
            pred_player_df.append(y_pred_rescaled[y])
        pred_player_df.append(position)
        append_df=pd.DataFrame([pred_player_df], columns=["Name", "p1","p2","p3","p4","p5","p6","p7","p8","position"])
        pred_all_players = pd.concat([pred_all_players, append_df], ignore_index=True)

    pred_all_players.to_csv(f"LSTM_{pred}.csv")

Generate_LSTM_preds("GOALS")

C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_22180\1565000828.py:9: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  opp_xg = data.apply(lambda row: row[22] if row[18] else row[20], axis=1)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_22180\1565000828.py:10: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  opp_xgc = data.apply(lambda row: row[23] if row[18] else row[21], axis=1)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_22180\1565000828.py:61: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consi

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 256ms/step


C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_22180\1565000828.py:115: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pred_all_players = pd.concat([pred_all_players, append_df], ignore_index=True)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_22180\1565000828.py:61: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  opp_xg = df.apply(lambda row: row[22] if row[18] else row[20], axis=1)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_22180\1565000828.py:62: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a f

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 245ms/step


C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_22180\1565000828.py:61: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  opp_xg = df.apply(lambda row: row[22] if row[18] else row[20], axis=1)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_22180\1565000828.py:62: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  opp_xgc = df.apply(lambda row: row[23] if row[18] else row[21], axis=1)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_22180\1565000828.py:63: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See

...

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 262ms/step


C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_22180\1565000828.py:61: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  opp_xg = df.apply(lambda row: row[22] if row[18] else row[20], axis=1)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_22180\1565000828.py:62: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  opp_xgc = df.apply(lambda row: row[23] if row[18] else row[21], axis=1)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_22180\1565000828.py:63: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 276ms/step


C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_22180\1565000828.py:61: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  opp_xg = df.apply(lambda row: row[22] if row[18] else row[20], axis=1)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_22180\1565000828.py:62: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  opp_xgc = df.apply(lambda row: row[23] if row[18] else row[21], axis=1)
C:\Users\OleJacobSimensen\AppData\Local\Temp\ipykernel_22180\1565000828.py:63: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 265ms/step
